# Gerar e corrigir um lote

O exemplo converte um XML de demonstração no modelo JSON, introduz um valor divergente e aplica a correção determinística.

In [1]:
from IPython.display import display

from voa_tiss import apply_fixes, build_lote, check, examples, parse_lote

lote = parse_lote(examples.load("lote-valido"))
item = lote["guias"][0]["procedimentos_executados"][0]
item["valor_total"] += 1

relatorio = check(lote)
achado = next(f for f in relatorio.findings if f.code == "item.valor_divergente")
display(achado.to_dict())

{'code': 'item.valor_divergente',
 'severity': 'error',
 'guia': '0001',
 'xpath': None,
 'pointer': '/guias/0/procedimentos_executados/0/valor_total',
 'message': 'Valor total do item diverge da quantidade, valor unitário e fator.',
 'fix': [{'op': 'test',
   'path': '/guias/0/procedimentos_executados/0/valor_total',
   'value': 10001},
  {'op': 'replace',
   'path': '/guias/0/procedimentos_executados/0/valor_total',
   'value': 10000}],
 'needs': None,
 'glosa': None,
 'rule': None}

O patch testa o valor atual antes de substituí-lo. A aplicação trabalha em uma cópia do lote.

In [2]:
corrigido = apply_fixes(lote, relatorio)
relatorio_final = check(corrigido)
xml = build_lote(corrigido)

display(
    {
        "valido": relatorio_final.valid,
        "codigos": [f.code for f in relatorio_final.findings],
        "round_trip": parse_lote(xml) == corrigido,
        "bytes_xml": len(xml),
    }
)

{'valido': True, 'codigos': [], 'round_trip': True, 'bytes_xml': 5018}